# Seattle Fault tsunami and ships


In [ ]:
from pathlib import Path
import hashlib
from importlib import reload
import json
import subprocess
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from shapely.geometry import box, shape
from shapely.ops import transform, unary_union
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'setrun.py').exists(): HERE = HERE / 'examples' / 'seattle_debris'
if not (HERE / 'setrun.py').exists(): raise FileNotFoundError('Run this notebook from the Seattle example or repository root')
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[1] / 'src'))
from geoclaw_debris import ship_tracking
reload(ship_tracking)
from geoclaw_debris.ship_tracking import Flow, Ship, hull_polygon, simulate

TSUNAMI_DIR = HERE / '_output' / 'seattle_fault'
TSUNAMI_FILE = TSUNAMI_DIR / 'seattle_fault_fgout.nc'
TERRAIN_FILE = TSUNAMI_DIR / 'seattle_buildings_mhw.tt3'
INVENTORY_FILE = HERE.parents[1] / 'SeattleBuildings' / 'Seattle_BuildingInventory.geojson'


## building heights

using `bldheight` in meters from SeattleBuildings/Seattle_BuildingInventory.geojson, retaining 20 m where it is missing


In [ ]:
from rasterio.features import rasterize
from rasterio.transform import from_origin
from clawpack.geoclaw import fgout_tools
from prepare_topo import read_mhw_region, write_topo, file_hash

def cached_flow(path, **attributes):
    if not path.exists(): return False
    with Dataset(path) as ds:
        return all(getattr(ds, key, None) == value for key, value in attributes.items())

def run_solver(folder):
    print('running GeoClaw; log:', folder / 'geoclaw.log', flush=True)
    with (folder / 'geoclaw.log').open('w') as log:
        subprocess.run([str(HERE / 'xgeoclaw')], cwd=folder, stdout=log, stderr=subprocess.STDOUT, check=True)

def save_flow(path, time_units='seconds', **attributes):
    grid = fgout_tools.FGoutGrid(1, str(path.parent), 'binary32')
    grid.read_fgout_grids_data()
    first = grid.read_frame(1)
    temporary = path.with_suffix('.partial.nc')
    with Dataset(temporary, 'w') as ds:
        ds.setncatts(dict(terrain_sha256=terrain_hash, inventory_sha256=inventory_hash,
                         building_height_source='NSI bldheight (m); missing heights use 20 m', **attributes))
        for name, values in [('lon', first.x), ('lat', first.y)]:
            ds.createDimension(name, len(values))
            ds.createVariable(name, 'f8', (name,))[:] = values
        ds.createDimension('time', grid.nout)
        ds.createVariable('time', 'f8', ('time',)).units = time_units
        for name in ('h', 'u', 'v', 'B'):
            var = ds.createVariable(name, 'f4', ('time', 'lon', 'lat'), zlib=True, complevel=1)
            var.units = 'm/s' if name in ('u', 'v') else 'm'
        for i in range(grid.nout):
            frame = first if i == 0 else grid.read_frame(i+1)
            ds['time'][i] = frame.t
            for name in ('h', 'u', 'v', 'B'): ds[name][i] = getattr(frame, name)
    temporary.replace(path)

features = json.loads(INVENTORY_FILE.read_text())['features']
inventory = pd.DataFrame([dict(geometry=shape(f['geometry']), nsi_height_m=f['properties'].get('bldheight'))
                          for f in features])
inventory['nsi_height_m'] = pd.to_numeric(inventory.nsi_height_m, errors='coerce')
inventory['height_m'] = inventory.nsi_height_m.where(inventory.nsi_height_m > 0, 20.)
TERRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
x, y, bare = read_mhw_region(HERE / 'puget_sound_mhw_19_n47x75_w122x50_2023v1.nc')
region = box(x[0], y[0], x[-1], y[-1])
local = inventory[inventory.geometry.map(lambda p: p.intersects(region))].sort_values('height_m')
transform_grid = from_origin(x[0]-(x[1]-x[0])/2, y[-1]+(y[1]-y[0])/2, x[1]-x[0], y[1]-y[0])
heights = rasterize(list(zip(local.geometry, local.height_m)),
                    out_shape=bare.shape, transform=transform_grid, fill=0, dtype='float64')[::-1]
terrain = np.where(heights > 0, np.maximum(bare, 0.)+heights, bare)
write_topo(TERRAIN_FILE, x, y, terrain)
terrain_hash = file_hash(TERRAIN_FILE)
inventory_hash = file_hash(INVENTORY_FILE)
print(f'{(local.nsi_height_m > 0).sum()}/{len(local)} footprints use NSI heights; others retain 20 m')


## Seattle Fault tsunami

instead of 12 m initial water ramp use the UW tsunami group's published
[SF-L Seattle Fault source](https://depts.washington.edu/ptha/dtopo/)

In [ ]:
from urllib.request import urlretrieve
from setrun import setrun, X1, X2, Y1, Y2

TSUNAMI_END = 600.
FINE_BOX = [-122.356, -122.340, 47.606, 47.615]
TSUNAMI_DIR.mkdir(parents=True, exist_ok=True)
source = TSUNAMI_DIR / 'seattlefault_uniform.tt3'
if not source.exists():
    temporary = source.with_suffix('.partial')
    urlretrieve('https://depts.washington.edu/ptha/dtopo/seattlefault_uniform.tt3', temporary)
    temporary.replace(source)
with Dataset(HERE / 'puget_sound_mhw_19_n47x75_w122x50_2023v1.nc') as ds:
    rx, ry = np.asarray(ds['lon'][::12]), np.asarray(ds['lat'][::12])
    regional = np.asarray(ds['z'][::12, ::12])
assert np.isfinite(regional).all()
write_topo(TSUNAMI_DIR / 'regional_mhw.tt3', rx, ry, regional)
run = setrun()
c = run.clawdata
dx, dy = (X2-X1)/349*16, (Y2-Y1)/314*16
c.lower = [float(X1-np.floor((X1-rx[0])/dx)*dx), float(Y1-np.floor((Y1-ry[0])/dy)*dy)]
c.num_cells = [int(np.floor((rx[-1]-c.lower[0])/dx)), int(np.floor((ry[-1]-c.lower[1])/dy))]
c.upper = [float(c.lower[0]+c.num_cells[0]*dx), float(c.lower[1]+c.num_cells[1]*dy)]
c.tfinal, c.output_style, c.output_times, c.output_t0, c.checkpt_style = TSUNAMI_END, 2, [TSUNAMI_END], False, 0
run.qinit_data.qinit_type, run.qinit_data.qinitfiles = 0, []
run.topo_data.topofiles = [[3, str(TSUNAMI_DIR / 'regional_mhw.tt3')], [3, str(TERRAIN_FILE)]]
run.dtopo_data.dtopofiles = [[3, str(source)]]
run.dtopo_data.dt_max_dtopo = .2
run.amrdata.amr_levels_max = 3
run.amrdata.refinement_ratios_x = run.amrdata.refinement_ratios_y = run.amrdata.refinement_ratios_t = [4, 4]
run.refinement_data.wave_tolerance = 1.e99
run.regiondata.regions = [[2, 2, 0., TSUNAMI_END, X1, X2, Y1, Y2],
                          [3, 3, 0., TSUNAMI_END, *FINE_BOX]]
fg = run.fgout_data.fgout_grids[0]
fg.nx, fg.ny, fg.tend, fg.nout = 349, 314, TSUNAMI_END, int(TSUNAMI_END/5)+1
signature = hashlib.sha256((terrain_hash + inventory_hash + file_hash(source)
             + file_hash(TSUNAMI_DIR / 'regional_mhw.tt3')
             + repr((c.lower, c.upper, c.num_cells, FINE_BOX, TSUNAMI_END))).encode()).hexdigest()
if not cached_flow(TSUNAMI_FILE, scenario_sha256=signature):
    run.write(str(TSUNAMI_DIR))
    run_solver(TSUNAMI_DIR)
    save_flow(TSUNAMI_FILE, time_units='seconds since earthquake', scenario_sha256=signature,
              forcing='UW SF-L Seattle Fault seafloor deformation; no qinit ramp',
              source_url='https://depts.washington.edu/ptha/dtopo/seattlefault_uniform.tt3', vertical_datum='MHW')


In [ ]:
with Dataset(TSUNAMI_FILE) as ds:
    flow = Flow(ds)
    domain = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
    selected = inventory[inventory.geometry.map(lambda p: p.intersects(domain))]
    footprints = list(selected.geometry)
    building_heights = selected.height_m.to_numpy()
    buildings = unary_union(footprints)
    metric_footprints = [transform(flow.to_xy, p) for p in footprints]
    metric_buildings = transform(flow.to_xy, buildings)
    reference = np.array(flow.to_xy(-122.3491, 47.61025))
    starts = [flow.to_lonlat(*(reference+offset)) for offset in [(0., 0.), (-45., 0.), (0., -35.), (-45., -35.)]]
    ships = [Ship(length_m=30., beam_m=8., draft_m=1.5) for _ in starts]
    print('running tsunami fleet', flush=True)
    paths = [simulate(flow, buildings, ship, start, start_time=10., end_time=TSUNAMI_END,
                      dt=1.) for ship, start in zip(ships, starts)]
results = [dict(case=dict(name=f'Ship {k+1}', start=start), ship=ship, sliding=path)
                   for k, (start, ship, path) in enumerate(zip(starts, ships, paths))]
hulls = [[hull_polygon((r['x_m'], r['y_m']), np.deg2rad(r['heading_deg']), ship) for r in path]
                 for ship, path in zip(ships, paths)]
assert all(not h.intersects(metric_buildings) for hulls in hulls for h in hulls)
for path in paths:
    lon, lat = flow.to_lonlat([r['x_m'] for r in path], [r['y_m'] for r in path])
    assert min(lon) >= FINE_BOX[0] and max(lon) <= FINE_BOX[1]
    assert min(lat) >= FINE_BOX[2] and max(lat) <= FINE_BOX[3]
display(pd.DataFrame([dict(ship=f'Ship {k+1}', final_status=path[-1]['status'],
        first_grounding_s=next((r['time_s'] for r in path if r['status']=='grounding'), np.nan),
        net_drift_m=round(np.hypot(path[-1]['x_m']-path[0]['x_m'], path[-1]['y_m']-path[0]['y_m']), 1))
        for k, path in enumerate(paths)]))


In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, LightSource, Normalize
from scipy.interpolate import RegularGridInterpolator
from shapely.geometry import Point, LineString
from IPython.display import HTML

def animate_ships(scene_results, scene_title, bounds):
    wet_depth = .1
    width = bounds[1]-bounds[0]
    ntracers = 180 if width > 300 else 100
    trail_seconds = 12.
    path = scene_results[0]['sliding']
    times = np.array([row['time_s'] for row in path])
    assert all(np.array_equal(times, [r['time_s'] for r in item['sliding']])
               for item in scene_results), 'Ships must share animation timestamps'
    origin = np.array([path[0]['x_m'], path[0]['y_m']])
    view = box(origin[0]+bounds[0], origin[1]+bounds[2], origin[0]+bounds[1], origin[1]+bounds[3])
    grid_x, grid_y = flow.to_xy(flow.lon, flow.lat)
    ix = (grid_x >= view.bounds[0]-20) & (grid_x <= view.bounds[2]+20)
    iy = (grid_y >= view.bounds[1]-20) & (grid_y <= view.bounds[3]+20)
    lon, lat = np.meshgrid(flow.lon[ix], flow.lat[iy], indexing='ij')
    x, y = flow.to_xy(lon, lat)
    with Dataset(TSUNAMI_FILE) as ds:
        water_times = np.asarray(ds['time'][:])
        frame = int(np.argmin(abs(water_times-times[0])))
        bottom = np.asarray(ds['B'][frame])[np.ix_(ix, iy)]
        fields = {name: np.asarray(ds[name][:, ix, :][:, :, iy]) for name in ('h', 'u', 'v')}

    field_functions = {
        name: RegularGridInterpolator((water_times, x[:, 0], y[0, :]), values,
                                      bounds_error=False, fill_value=np.nan)
        for name, values in fields.items()
    }
    def sample_water(name, points, time):
        return field_functions[name](np.column_stack((np.full(len(points), time), points)))

    grid = np.column_stack((x.ravel(), y.ravel()))
    def water_grid(time): return sample_water('h', grid, time).reshape(x.shape)
    def current(points, time):
        return np.column_stack([sample_water(name, points, time) for name in ('u', 'v')])

    visible_buildings = unary_union([p for p in metric_footprints if p.intersects(view)])
    def wet_open(points, time):
        depth = sample_water('h', points, time)
        return np.array([np.isfinite(d) and d >= wet_depth and view.contains(Point(p))
                         and not visible_buildings.intersects(Point(p))
                         for p, d in zip(points, depth)])

    rng = np.random.default_rng(7)
    def release_particles(count, time):
        candidates = rng.uniform([view.bounds[0], view.bounds[1]],
                                 [view.bounds[2], view.bounds[3]], size=(max(500, count*20), 2))
        candidates = candidates[wet_open(candidates, time)]
        if not len(candidates): return np.full((count, 2), np.nan)
        return candidates[rng.choice(len(candidates), size=count, replace=len(candidates) < count)]

    particle_xy = np.full((len(times), ntracers, 2), np.nan)
    particle_xy[0] = release_particles(ntracers, times[0])
    particle_generation = np.zeros((len(times), ntracers), dtype=int)
    for k in range(1, len(times)):
        positions = particle_xy[k-1].copy()
        generation = particle_generation[k-1].copy()
        steps = max(1, int(np.ceil((times[k]-times[k-1])/.5)))
        step = (times[k]-times[k-1])/steps
        for j in range(steps):
            time = times[k-1] + j*step
            velocity = current(positions, time)
            midpoint = positions + .5*step*velocity
            mid_velocity = current(midpoint, time+.5*step)
            proposed = positions + step*mid_velocity
            valid = wet_open(midpoint, time+.5*step) & wet_open(proposed, time+step)
            for q in np.flatnonzero(valid):
                valid[q] = not LineString([positions[q], midpoint[q], proposed[q]]).intersects(visible_buildings)
            positions[valid] = proposed[valid]
            positions[~valid] = release_particles(np.count_nonzero(~valid), time+step)
            generation[~valid] += 1
        particle_xy[k] = positions
        particle_generation[k] = generation

    fig, ax = plt.subplots(figsize=(12, 8))
    cmap = ListedColormap(plt.get_cmap('Blues')(np.linspace(.15, .8, 256)))
    ax.set_facecolor('#eeeae2')
    land_cmap = ListedColormap(plt.get_cmap('gist_earth')(np.linspace(.3, 1., 256)))
    elevation = Normalize(0, 80, clip=True)
    dx, dy = x[1, 0]-x[0, 0], y[0, 1]-y[0, 0]
    relief = LightSource(azdeg=315, altdeg=45).shade(bottom.T, cmap=land_cmap,
                  norm=elevation, dx=dx, dy=dy, vert_exag=1., blend_mode='soft')
    ax.imshow(relief, origin='lower', interpolation='bilinear', extent=(x.min()-origin[0]-dx/2, x.max()-origin[0]+dx/2,
              y.min()-origin[1]-dy/2, y.max()-origin[1]+dy/2), zorder=0)
    contours = ax.contour(x-origin[0], y-origin[1], bottom, levels=[-20, -10, 0, 5, 10, 20, 40, 60, 80],
                          colors='#555044', linewidths=.4, alpha=.5, zorder=1.2)
    ax.clabel(contours, fontsize=6, fmt='%g', inline_spacing=3)
    depth = water_grid(times[0])
    water = ax.pcolormesh(x-origin[0], y-origin[1], np.ma.masked_where(depth < wet_depth, depth),
                          cmap=cmap, vmin=0, vmax=12, shading='auto', alpha=.7, zorder=1)
    fig.colorbar(water, ax=ax, label='water depth (m)', shrink=.75, pad=.02)
    fig.colorbar(plt.cm.ScalarMappable(norm=elevation, cmap=land_cmap), ax=ax,
                 label='ground / roof elevation (m MHW)', shrink=.75, pad=.08, extend='max')
    for footprint, height in zip(metric_footprints, building_heights):
        if not footprint.intersects(view): continue
        pieces = footprint.geoms if hasattr(footprint, 'geoms') else [footprint]
        for piece in pieces:
            if piece.geom_type == 'Polygon':
                ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords)-origin,
                             facecolor='none', edgecolor='#45483f', lw=.8, zorder=4))

        if width <= 100 and view.contains(footprint.centroid):
            center = np.array(footprint.centroid.coords[0])-origin
            ax.text(*center, f'{height:.1f} m', ha='center', fontsize=8, color='#222', zorder=5)

    stride = max(3, round(width/15 / (x[1, 0]-x[0, 0])))
    arrow_xy = np.column_stack((x[::stride, ::stride].ravel(), y[::stride, ::stride].ravel()))
    def arrow_velocity(time):
        velocity = current(arrow_xy, time)
        return np.ma.masked_where(np.repeat(~wet_open(arrow_xy, time), 2).reshape(-1, 2), velocity)
    velocity = arrow_velocity(times[0])
    arrows = ax.quiver(*(arrow_xy-origin).T, velocity[:, 0], velocity[:, 1],
                       color='#36576b', angles='xy', scale_units='xy', scale=40/width,
                       width=.003, alpha=.75, zorder=2)
    ax.quiverkey(arrows, .82, 1.015, 2., '2 m/s', labelpos='E', coordinates='axes')
    particle_trails = LineCollection([], colors='#ffffff', linewidths=1., alpha=.7, zorder=3)
    ax.add_collection(particle_trails)
    particles = ax.scatter(*(particle_xy[0]-origin).T, s=8, color='white', edgecolors='#36576b',
                           linewidths=.25, zorder=3, label='current tracers')
    hulls, ship_trails, labels = [], [], []
    colors = ['#d65a3a', '#c68b16', '#188b81', '#9465b0']
    for k, item in enumerate(scene_results):
        row, vessel = item['sliding'][0], item['ship']
        outline = hull_polygon((row['x_m'], row['y_m']), np.deg2rad(row['heading_deg']), vessel)
        hull = PolygonPatch(np.asarray(outline.exterior.coords)-origin, facecolor=colors[k % len(colors)],
                            edgecolor='#222', lw=1.5, zorder=6)
        ax.add_patch(hull)
        hulls.append(hull)
        ship_trails.append(ax.plot([], [], color=colors[k % len(colors)], lw=1.8, zorder=5,
                                   label=item['case']['name'])[0])
        labels.append(ax.text(row['x_m']-origin[0], row['y_m']-origin[1], str(k+1),
                              ha='center', va='center', color='white', weight='bold', zorder=7))
    ax.set(xlim=bounds[:2], ylim=bounds[2:], xlabel='east of reference launch (m)', ylabel='north of reference launch (m)')
    ax.set_aspect('equal')
    ax.annotate('N', xy=(.95, .94), xytext=(.95, .86), xycoords='axes fraction',
                ha='center', arrowprops=dict(arrowstyle='->', color='#333'))

    def update(i):
        time = times[i]
        depth = water_grid(time)
        water.set_array(np.ma.masked_where(depth < wet_depth, depth).ravel())
        velocity = arrow_velocity(time)
        arrows.set_UVC(velocity[:, 0], velocity[:, 1])
        particles.set_offsets(particle_xy[i]-origin)
        first = np.searchsorted(times, time-trail_seconds)
        segments = []
        for k in range(first, i):
            valid = particle_generation[k] == particle_generation[k+1]
            segments.extend(np.stack((particle_xy[k, valid], particle_xy[k+1, valid]), axis=1)-origin)
        particle_trails.set_segments(segments)
        statuses = []
        for k, item in enumerate(scene_results):
            row = item['sliding'][i]
            outline = hull_polygon((row['x_m'], row['y_m']), np.deg2rad(row['heading_deg']), item['ship'])
            hulls[k].set_xy(np.asarray(outline.exterior.coords)-origin)
            ship_trails[k].set_data([r['x_m']-origin[0] for r in item['sliding'][:i+1]],
                                   [r['y_m']-origin[1] for r in item['sliding'][:i+1]])
            labels[k].set_position((row['x_m']-origin[0], row['y_m']-origin[1]))
            statuses.append(f"{k+1}: {row['status']}")
        ax.set_title(f"{scene_title}: {time:.0f} s\n" + ' | '.join(statuses), fontsize=10)

    frames = sorted(set(range(0, len(path), max(2, int(np.ceil(len(path)/85))))) | {len(path)-1})
    update(0)
    fig.tight_layout()
    anim = FuncAnimation(fig, update, frames=frames, interval=120)
    plt.close(fig)
    return anim

plt.rcParams['animation.embed_limit'] = 60


In [ ]:
vertices = np.vstack([np.asarray(h.exterior.coords) for hulls in hulls for h in hulls])
origin = np.array([paths[0][0]['x_m'], paths[0][0]['y_m']])
vertices -= origin
bounds = (min(-300, vertices[:, 0].min()-60), max(300, vertices[:, 0].max()+60),
          min(-220, vertices[:, 1].min()-60), max(280, vertices[:, 1].max()+60))
anim = animate_ships(results, 'Seattle Fault tsunami', bounds=bounds)
HTML(anim.to_jshtml())
